In [1]:
import json
import shutil
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from pathlib import Path

TypeError: _TypedDictMeta.__new__() got an unexpected keyword argument 'extra_items'

In [2]:
sections_dir = Path("../data/json/security")
sections = []

for json_file in sorted(sections_dir.glob("*.json")):
    file_sections = json.loads(json_file.read_text(encoding="utf-8"))
    sections.extend(file_sections)
    print(f"Loaded {len(file_sections):>3} sections  ←  {json_file.name}")

print(f"\nTotal sections loaded: {len(sections)}")

NameError: name 'Path' is not defined

In [ ]:
from itertools import groupby

CHUNK_SIZE            = 3
OVERLAP               = 1
STRIDE                = CHUNK_SIZE - OVERLAP   # = 2
MAX_CHARS_PER_SECTION = 800

# Boilerplate headings that appear identically in every Spring guide — skip them
SKIP_HEADINGS = {
    "What you'll build",
    "How to complete this guide",
    "What you'll need",
    "Summary"
}

chunks = []

# Sort ensures groupby works even if sections from different files are interleaved
sorted_sections = sorted(sections, key=lambda s: s["source"])

for source, group in groupby(sorted_sections, key=lambda s: s["source"]):
    doc_sections = [
        s for s in group
        if s["content"].strip() and s.get("heading", "") not in SKIP_HEADINGS
    ]
    for i in range(0, len(doc_sections), STRIDE):
        window = doc_sections[i : i + CHUNK_SIZE]
        content = "\n\n".join(
            (f"## {s['heading']}\n" if s["heading"] else "") +
            s["content"].strip()[:MAX_CHARS_PER_SECTION]
            for s in window
        )
        if not content.strip():
            continue
        chunks.append(Document(
            page_content=content,
            metadata={
                "headings": " | ".join(s["heading"] for s in window if s["heading"]),
                "title":    window[0]["title"],
                "source":   source,
            }
        ))

print(f"Created {len(chunks)} chunks  ({CHUNK_SIZE} sections/chunk, {OVERLAP} overlap)")

Created 59 chunks  (3 sections/chunk, 1 overlap)


In [4]:
import ollama

response = ollama.embed(
    model='nomic-embed-text',
    input=chunks[0].page_content,
)
print(f"{response.embeddings} : {chunks[0].metadata['source']}")

[[0.059809543, 0.04631697, -0.17938273, -0.03482624, 0.023362612, -0.05550568, 0.0103470385, 0.0068843463, -0.024365645, -0.04786128, -0.04293048, 0.009278206, 0.06924877, -0.015894653, -0.008680033, -0.0022421603, -0.009715727, -0.030771593, -0.02002831, 0.028440665, -0.0015772165, -0.061708536, 0.00063719647, -0.035681553, 0.057113457, 0.061881118, -0.033294097, 0.0059863897, -0.02911948, 0.030741237, 0.01833253, -0.017501842, -0.028734377, 0.007935214, -0.06358074, -0.052201916, -0.03281, -0.031089555, -0.011858279, -0.011732987, 0.042396598, -0.040474687, 0.02443701, -0.053519588, 0.04720544, 0.010829379, 0.0809627, -0.0062440243, 0.032916598, -0.045276556, -0.0073358933, 0.017237501, -0.010587336, -0.008968934, 0.02742132, 0.032852203, 0.047485236, 0.00021279056, -0.0033542267, -0.043565117, 0.14400771, 0.0551298, -0.027958306, 0.048337825, 0.05807851, 5.985787e-05, 0.003912028, 0.024502572, -0.008483917, -0.02374343, 0.026703874, -0.016709695, 0.024117492, -0.055235855, -0.031036

In [ ]:
# Clear old Chroma DB before re-embedding to avoid duplicates
shutil.rmtree("./chroma_db/security", ignore_errors=True)
print("Cleared old vector store.")

Cleared old vector store.


In [ ]:
# Embed chunks and store in Chroma (persisted to disk)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectors_dir = "./chroma_db/security"

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=vectors_dir,
    collection_name="spring_security"
)

print(f"Stored {vector_store._collection.count()} embeddings in '{vectors_dir}'")

Stored 59 embeddings in './chroma_db/data_access'


In [ ]:
# Sanity check: run a test similarity search
query = "Authentication with Spring Security"
results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source : {doc.metadata.get('source', 'unknown')}")
    print(f"Page   : {doc.metadata.get('page', '?')}")
    print(f"Content: {doc.page_content[:100]}")
    print()

--- Result 1 ---
Source : https://spring.io/guides/gs/accessing-data-neo4j
Page   : ?
Content: ## Create an Application Class
Spring Initializr creates a simple class for the application. The fol

--- Result 2 ---
Source : https://spring.io/guides/gs/spring-data-reactive-redis
Page   : ?
Content: ## How to Complete This Guide
Like most Spring Getting Started guides you can start from scratch and

--- Result 3 ---
Source : https://spring.io/guides/gs/accessing-data-neo4j
Page   : ?
Content: ## Create Simple Queries
Spring Data Neo4j is focused on storing data in Neo4j. But it inherits func

